In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense , Flatten
import matplotlib.pyplot as plt

In [ ]:
(X_train,y_train),(X_test,y_test) = keras.datasets.mnist.load_data()
X_train


In [ ]:
print('Training set shape', X_train.shape)
print('Test set shape', X_test.shape)
print('Training labels shape', y_train.shape)

In [ ]:
# VISUALIZING SAMPLE IMAGES
plt.figure(figsize = (12,4))
for i in range(5):
    plt.subplot(1,5,i+1)
    plt.imshow(X_train[i],cmap = 'gray')
    plt.title(f'Label : {y_train[i]}')
    plt.axis('off')

In [ ]:
print('Original pixel range: ', X_train.min(), 'to', X_train.max())
print(X_train[0])

#### Data Preprocessing

In [ ]:
# Step 1 : Normalization (Scaling)
X_train = X_train/255.0
X_test = X_test/255.0

In [ ]:
print('Normalized range: ', X_train.min(), 'to', X_train.max())

##### UNDERSTANDING THE DATA SHAPES


In [ ]:
# CURRENT DATA STRUCTURE
print('Current X_train shape : ',X_train.shape)
print('Current y_train shape : ', y_train.shape)


# Sample of normalized data
print('sMPLE NORMALIZED PIXEL VALUES.')
print(X_train[0][10:15, 10:15])

#### BUILDING THE MODEL

In [ ]:
# STEP 1 : Create sequential Model
model = Sequential()


In [ ]:
# STEP 2: ADD FLATTEN LAYER
model.add(Flatten(input_shape=(28,28)))


In [ ]:
# STEP 3 : ADD HIDDEN LAYERS
# FIRTS HIDDEN LAYER
model.add(Dense(128,activation='relu'))

In [ ]:
# SECOND HIDDEN LAYER
model.add(Dense(32,activation = 'relu'))

In [ ]:
# STEP 4 : ADD OUTPUT LAYER
model.add(Dense(10,activation='softmax'))

In [ ]:
model.summary()

#### MODEL TRAINING

In [ ]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer = 'Adam',
    metrics = ['accuracy']
)

In [ ]:
history = model.fit(X_train,y_train, epochs = 25, validation_split = 0.2, verbose = 1)

#### MODEL EVALUATION

In [ ]:
# STEP 1 : MAKE PREDICTIONS 
y_prob = model.predict(X_test)
print('Prediction shape : ', y_prob.shape)
print('Sample probabilities for first image.')
print(y_prob[0])

In [ ]:
y_pred = y_prob.argmax(axis = 1)
print('Predicted classes :', y_pred[:10])
print('Actual classes : ', y_test[:10])

In [ ]:
# Step 3 : calculate accuracy
from sklearn.metrics import accuracy_score
test_accuracy = accuracy_score(y_test,y_pred)
print(f'Test Accuracy  : {test_accuracy:.4f}')

#### Visualization and Analysis

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize = (12,4))
  # Loss plot
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label  = 'Training loss', color = 'blue')
plt.plot(history.history['val_loss'], label = 'Validation Loss', color = 'orange')
plt.title('Model Loss Over Time')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha = 0.3)


# Accuracy Plot
plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label = 'Training Accuracy', color = 'green')
plt.plot(history.history['val_accuracy'], label = 'Validation Accuracy', color = 'red')
plt.title('Model Accuracy Over Time')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha  = 0.3)
plt.tight_layout()
plt.show()

In [ ]:
def test_prediction(index):
    # Display image
    plt.figure(figsize=(8, 3))

    plt.subplot(1, 2, 1)
    plt.imshow(X_test[index], cmap='gray')
    plt.title(f'Actual: {y_test[index]}')
    plt.axis('off')

    # Show prediction probabilities
    plt.subplot(1, 2, 2)

    image = X_test[index].reshape(1, 28, 28)

    probs = model.predict(image, verbose=0)[0]

    plt.bar(range(10), probs)
    plt.title(f'Predicted: {probs.argmax()}')
    plt.xlabel('Digit Class')
    plt.ylabel('Probability')
    plt.xticks(range(10))

    plt.tight_layout()
    plt.show()


# Test first few images
for i in range(3):
    test_prediction(i)

#### Model Improvements

In [ ]:
# Add more hidden layers
model_improved = Sequential([ 
    Flatten(input_shape=(28, 28)), 
    Dense(256, activation='relu'),
    Dense(128, activation='relu'), 
    Dense(64, activation='relu'), 
    Dense(32, activation='relu'), 
    Dense(10, activation='softmax') 
])


# Experiment with layers sizes
# Different configurations to try: 
configs = [
[128, 64, 32], 
[64, 128, 64], 
[256, 256, 128], 
[512, 256, 128, 64] 
]



# Add regularization
from tensorflow.keras.layers import Dropout
model_regularized = Sequential([ 
    Flatten(input_shape=(28, 28)), 
    Dense(128, activation='relu'), 
    Dropout(0.2), 
    Dense(64, activation='relu'), 
    Dropout(0.2), 
    Dense(10, activation='softmax') 
])


# Longer Training
# Train for more epochs 
history_long = model.fit(
    X_train, y_train, 
    epochs=50, 
    validation_split=0.2, 
    batch_size=32 
)



# Different Optimizers
# Try different optimizers 
optimizers_to_try = [
'adam',
'sgd', 
'rmsprop', 
' adagrad'
]
for opt in optimizers_to_try:
model.compile(
    loss ='sparse_categorical_crossentropy', 
    optimizer=opt,
    metrics=['accuracy'] 
)


In [ ]:
# Understanding Multi-Class Output
# Softmax ensures probabilities sum to 1 
sample_output = model.predict(X_test[0].reshape(1, 28, 28))[0] 
print("Individual probabilities:") 
for i, prob in enumerate(sample_output):
    print(f"Digit {i}: {prob:.4f}")
print(f"Sum of probabilities: {sample_output.sum():.4f}") # Should be 1.0

In [ ]:
# Confusion Matrix Analysis

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Generate predictions
y_pred = model.predict(X_test, verbose=0).argmax(axis=1)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.title('Confusion Matrix - MNIST Digit Classification')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

plt.show()

# Detailed classification report
print(classification_report(y_test, y_pred))

In [ ]:
# Error Analysis
# Find misclassified examples

import numpy as np
import matplotlib.pyplot as plt

errors = y_test != y_pred
error_indices = np.where(errors)[0]

print(f"Total errors: {errors.sum()}")
print(f"Error rate: {errors.mean():.4f}")

# Display some misclassified examples
plt.figure(figsize=(15, 3))

for i, idx in enumerate(error_indices[:5]):
    plt.subplot(1, 5, i + 1)
    plt.imshow(X_test[idx], cmap='gray')
    plt.title(f"True: {y_test[idx]}, Pred: {y_pred[idx]}")
    plt.axis('off')

plt.tight_layout()
plt.show()